In [1]:
import os
import pandas as pd
import numpy as np
import datetime as dt
import math
from tqdm import tqdm

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-12-17 22:40:15.960296


#### Functions

#### Constants

In [3]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# subtask
str_subtask = os.getcwd().split('/')[6]
print(f'Subtask: {str_subtask}')

# get the dob
int_dob = int(str_task[3:].split('_')[-4])
print(f'DOB: {int_dob}')

# logic
if int_dob == 720:
    str_target = 'Early_Pay_Delinquency_60_720_Flag'
else:
    pass
print(f'Target: {str_target}')

str_dirname_output = './output'

flt_prop_train = 0.4

flt_prop_valid = 0.2

flt_prop_test = 0.4

int_n_months_inform = 2

int_n_months_holdout = 3

Project: 20241112-simple-model-test
Task: 16_60_in_720_dl_loss_test
Subtask: 01_data_split
DOB: 720
Target: Early_Pay_Delinquency_60_720_Flag


#### Make output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [5]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/08_prep_data/{str_filename}'
df = pd.read_parquet(
    str_uri,
)

# get month of request
df['request_month'] = df['request_datetime'].apply(
    lambda x: f'{str(x)[:7]}-01',
)
df['request_month'] = pd.to_datetime(df['request_month']).dt.date
# sort
df.sort_values(by='request_month', ascending=True, inplace=True)

# show
df

CPU times: user 10.2 s, sys: 3.85 s, total: 14.1 s
Wall time: 6.67 s


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,loss_at_180,loss_at_360,loss_at_720,ENG-franchise,ENG-has_codebtor,ENG-vehicle_age,ENG-payment_to_income,ENG-loan_to_value,ENG-bk,request_month
175,5716209,2021-07-30 13:53:28.0444771,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Washington,Franchise,Washington,True,...,0.0,0.0,0.0,1,0,3,NaN,1.444396,0,2021-07-01
154,5702150,2021-07-30 10:22:57.8246513,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Indiana,Franchise,Indiana,True,...,0.0,0.0,0.0,1,0,4,NaN,0.938360,1,2021-07-01
18,5702106,2021-07-27 12:40:50.0115498,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,California,Franchise,California,False,...,0.0,0.0,0.0,1,0,2,NaN,1.347324,0,2021-07-01
207,5714824,2021-07-30 17:16:33.7898586,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Illinois,Franchise,Illinois,False,...,0.0,0.0,0.0,1,1,7,0.113477,1.174497,0,2021-07-01
206,5714824,2021-07-30 17:16:33.7898586,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,False,...,0.0,0.0,0.0,1,1,7,0.105172,1.174497,0,2021-07-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93346,8359905,2024-11-07 04:33:53+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-06.gzip,0,0,Illinois,Franchise,Indiana,True,...,0.0,0.0,0.0,1,1,-1,NaN,1.221521,0,2024-11-01
93345,8359905,2024-11-07 04:33:53+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-06.gzip,1,1,Illinois,Franchise,Indiana,True,...,0.0,0.0,0.0,1,1,-1,0.139565,1.221521,0,2024-11-01
93643,8359885,2024-11-12 23:23:03+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-12.gzip,1,1,Indiana,Franchise,Indiana,True,...,0.0,0.0,0.0,1,0,4,0.083187,1.290518,0,2024-11-01
93286,8359691,2024-11-06 06:11:41+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-05.gzip,1,1,Missouri,Franchise,Missouri,False,...,0.0,0.0,0.0,1,0,2,0.093134,1.120000,0,2024-11-01


#### Create target

In [6]:
%%time

df[str_target] = df.apply(
    lambda x: 1 if (x[str_target] == 1) and (x[f'loss_at_{int_dob}'] > 0) else 0,
    axis=1,
)

CPU times: user 11 s, sys: 5.68 s, total: 16.7 s
Wall time: 16.6 s


#### Remove too new from days on books (funded date - run date)

In [7]:
%%time

df = df[df['days_on_books'] >= int_dob].copy()
# show
df

CPU times: user 530 ms, sys: 545 ms, total: 1.07 s
Wall time: 1.07 s


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,loss_at_180,loss_at_360,loss_at_720,ENG-franchise,ENG-has_codebtor,ENG-vehicle_age,ENG-payment_to_income,ENG-loan_to_value,ENG-bk,request_month
175,5716209,2021-07-30 13:53:28.0444771,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Washington,Franchise,Washington,True,...,0.0,0.00,0.0,1,0,3,NaN,1.444396,0,2021-07-01
154,5702150,2021-07-30 10:22:57.8246513,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Indiana,Franchise,Indiana,True,...,0.0,0.00,0.0,1,0,4,NaN,0.938360,1,2021-07-01
18,5702106,2021-07-27 12:40:50.0115498,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,California,Franchise,California,False,...,0.0,0.00,0.0,1,0,2,NaN,1.347324,0,2021-07-01
207,5714824,2021-07-30 17:16:33.7898586,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Illinois,Franchise,Illinois,False,...,0.0,0.00,0.0,1,1,7,0.113477,1.174497,0,2021-07-01
206,5714824,2021-07-30 17:16:33.7898586,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,False,...,0.0,0.00,0.0,1,1,7,0.105172,1.174497,0,2021-07-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42089,6464887,2022-12-06 22:54:50+00:00,PRESTIGE-GENXI,01_pull_payloads/df_requests_2022-12-06.gzip,1,1,Illinois,Franchise,Illinois,True,...,0.0,0.00,0.0,1,0,3,NaN,1.456915,1,2022-12-01
42197,6464638,2022-12-07 23:13:46+00:00,PRESTIGE-GENXI,01_pull_payloads/df_requests_2022-12-07.gzip,1,1,Virginia,Independent,Virginia,False,...,0.0,0.00,0.0,0,0,7,NaN,1.379535,0,2022-12-01
41923,6464416,2022-12-03 00:37:04+00:00,PRESTIGE-GENXI,01_pull_payloads/df_requests_2022-12-02.gzip,1,1,Utah,Independent,Utah,False,...,0.0,0.00,0.0,0,0,6,0.167883,1.350251,0,2022-12-01
42062,6465473,2022-12-06 07:13:43+00:00,PRESTIGE-GENXI,01_pull_payloads/df_requests_2022-12-06.gzip,1,1,Alabama,Franchise,Alabama,False,...,0.0,0.00,0.0,1,1,3,NaN,1.374376,1,2022-12-01


#### Get the request months

In [8]:
list_request_month = list(df['request_month'].value_counts().index)
# sort
list_request_month = sorted(list_request_month)

# show
for a, request_month in enumerate(list_request_month):
    print(f'{a+1} - {request_month}')

1 - 2021-07-01
2 - 2021-08-01
3 - 2021-09-01
4 - 2021-10-01
5 - 2021-11-01
6 - 2021-12-01
7 - 2022-01-01
8 - 2022-02-01
9 - 2022-03-01
10 - 2022-04-01
11 - 2022-05-01
12 - 2022-06-01
13 - 2022-07-01
14 - 2022-08-01
15 - 2022-09-01
16 - 2022-10-01
17 - 2022-11-01
18 - 2022-12-01


#### Get the training months

In [9]:
int_n_request_months = len(list_request_month)
# subtract inform and holdout
int_n_training_months = int_n_request_months - int_n_months_holdout - int_n_months_inform

# get the number of months for training data set
int_n_months_training_data = math.ceil(int_n_training_months * flt_prop_train)
# get the months
list_request_months_training = list_request_month[:int_n_months_training_data]

# show
for a, request_month in enumerate(list_request_months_training):
    print(f'{a+1} - {request_month}')

1 - 2021-07-01
2 - 2021-08-01
3 - 2021-09-01
4 - 2021-10-01
5 - 2021-11-01
6 - 2021-12-01


#### Get the validation months

In [10]:
# get the number of months for validation data set
int_n_months_validation_data = math.ceil(int_n_training_months * flt_prop_valid)
int_end_tmp = int_n_months_training_data + int_n_months_validation_data
# get the months
list_request_months_validation = list_request_month[int_n_months_training_data:int_end_tmp]

# show
for a, request_month in enumerate(list_request_months_validation):
    print(f'{a+1} - {request_month}')

1 - 2022-01-01
2 - 2022-02-01
3 - 2022-03-01


#### Get the test months

In [11]:
int_start_tmp = int_n_months_training_data + int_n_months_validation_data
int_n_months_test_data = int_n_training_months - int_start_tmp
# get the months
list_request_months_test = list_request_month[int_start_tmp:int_start_tmp+int_n_months_test_data]

# show
for a, request_month in enumerate(list_request_months_test):
    print(f'{a+1} - {request_month}')

1 - 2022-04-01
2 - 2022-05-01
3 - 2022-06-01
4 - 2022-07-01


#### Get the inform months

In [12]:
int_start_tmp = int_n_months_training_data + int_n_months_validation_data + int_n_months_test_data
# get the months
list_request_months_inform = list_request_month[int_start_tmp:int_start_tmp+int_n_months_inform]

# show
for a, request_month in enumerate(list_request_months_inform):
    print(f'{a+1} - {request_month}')

1 - 2022-08-01
2 - 2022-09-01


#### Get the holdout months

In [13]:
int_start_tmp = int_n_months_training_data + int_n_months_validation_data + int_n_months_test_data + int_n_months_inform
# get the months
list_request_months_holdout = list_request_month[int_start_tmp:]

# show
for a, request_month in enumerate(list_request_months_holdout):
    print(f'{a+1} - {request_month}')

1 - 2022-10-01
2 - 2022-11-01
3 - 2022-12-01


#### Create tags

In [14]:
%%time

# train
df['train'] = df['request_month'].apply(
    lambda x: 1 if x in list_request_months_training else 0,
)

CPU times: user 23.9 ms, sys: 3.99 ms, total: 27.9 ms
Wall time: 26.9 ms


In [15]:
%%time

# valid
df['valid'] = df['request_month'].apply(
    lambda x: 1 if x in list_request_months_validation else 0,
)

CPU times: user 28.5 ms, sys: 295 μs, total: 28.8 ms
Wall time: 28.5 ms


In [16]:
%%time

# test
df['test'] = df['request_month'].apply(
    lambda x: 1 if x in list_request_months_test else 0,
)

CPU times: user 34.5 ms, sys: 0 ns, total: 34.5 ms
Wall time: 33.3 ms


In [17]:
%%time

# test
df['inform'] = df['request_month'].apply(
    lambda x: 1 if x in list_request_months_inform else 0,
)

CPU times: user 24.6 ms, sys: 0 ns, total: 24.6 ms
Wall time: 23.8 ms


In [18]:
%%time

# test
df['holdout'] = df['request_month'].apply(
    lambda x: 1 if x in list_request_months_holdout else 0,
)

CPU times: user 25 ms, sys: 490 μs, total: 25.5 ms
Wall time: 24.6 ms


#### Write to s3

In [19]:
list_str_filename = [
    'train',
    'valid',
    'test',
    'inform',
    'holdout',
]
for str_filename in tqdm(list_str_filename):
    print(f'Filename: {str_filename}')
    # subset
    df_tmp = df[df[str_filename] == 1].copy()
    # get n rows
    int_nrows = df_tmp.shape[0]
    print(f'Rows: {int_nrows}')
    print()
    # make column name
    df_tmp['data_set'] = str_filename
    # write
    str_filename_tmp = f'df_{str_filename}.gzip'
    str_uri = f's3://{str_project}/{str_task}/{str_subtask}/{str_filename_tmp}'
    df_tmp.to_parquet(str_uri, compression='gzip')

  0%|          | 0/5 [00:00<?, ?it/s]

Filename: train
Rows: 9304



 20%|██        | 1/5 [00:03<00:14,  3.50s/it]

Filename: valid
Rows: 6305



 40%|████      | 2/5 [00:07<00:12,  4.06s/it]

...

 60%|██████    | 3/5 [00:14<00:10,  5.11s/it]

Filename: inform
Rows: 6679



 80%|████████  | 4/5 [00:18<00:04,  4.75s/it]

Filename: holdout
Rows: 7799



100%|██████████| 5/5 [00:22<00:00,  4.50s/it]
